In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Read CSV Files').getOrCreate()

In [3]:
# Deklarasi path data disimpan
path_block0 = "/content/sample_data/block_0.csv"
path_weather = "/content/sample_data/weather_hourly_darksky.csv"

# Membaca berkas secara terpisah tanpa inferSchema agar proses pemuatan berlangsung instan
df_block0 = spark.read.csv(path_block0, header=True, sep=',')
df_weather = spark.read.csv(path_weather, header=True, sep=',')

In [4]:
# Penggunaan .limit(10) memastikan hanya 10 baris yang ditarik
display(df_block0.limit(10).toPandas())
display(df_weather.limit(10).toPandas())

,LCLid,tstp,energy(kWh/hh)
0,MAC000002,2012-10-12 00:30:00.0000000,0
1,MAC000002,2012-10-12 01:00:00.0000000,0
2,MAC000002,2012-10-12 01:30:00.0000000,0
3,MAC000002,2012-10-12 02:00:00.0000000,0
4,MAC000002,2012-10-12 02:30:00.0000000,0
5,MAC000002,2012-10-12 03:00:00.0000000,0
6,MAC000002,2012-10-12 03:30:00.0000000,0
7,MAC000002,2012-10-12 04:00:00.0000000,0
8,MAC000002,2012-10-12 04:30:00.0000000,0
9,MAC000002,2012-10-12 05:00:00.0000000,0


,visibility,windBearing,temperature,time,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,5.97,104,10.24,2011-11-11 00:00:00,8.86,1016.76,10.24,2.77,rain,partly-cloudy-night,0.91,Partly Cloudy
1,4.88,99,9.76,2011-11-11 01:00:00,8.83,1016.63,8.24,2.95,rain,partly-cloudy-night,0.94,Partly Cloudy
2,3.7,98,9.46,2011-11-11 02:00:00,8.79,1016.36,7.76,3.17,rain,partly-cloudy-night,0.96,Partly Cloudy
3,3.12,99,9.23,2011-11-11 03:00:00,8.63,1016.28,7.44,3.25,rain,fog,0.96,Foggy
4,1.85,111,9.26,2011-11-11 04:00:00,9.21,1015.98,7.24,3.7,rain,fog,1.0,Foggy
5,1.96,115,9.33,2011-11-11 05:00:00,8.87,1015.91,7.19,3.97,rain,fog,0.97,Foggy
6,1.3,118,9.31,2011-11-11 06:00:00,8.82,1015.7,7.1,4.1,rain,fog,0.97,Foggy
7,1.22,114,8.85,2011-11-11 07:00:00,8.69,1016.08,6.48,4.23,rain,fog,0.99,Foggy
8,1.4,120,9.13,2011-11-11 08:00:00,8.75,1016.33,6.84,4.2,rain,fog,0.97,Foggy
9,1.38,121,9.23,2011-11-11 09:00:00,8.7,1016.57,7.07,3.96,rain,fog,0.97,Foggy


In [9]:
from pyspark.sql.functions import to_timestamp, date_trunc, sum as _sum, count, col, expr

# Filter baris yang mengandung string 'Null', 'null', atau bernilai kosong
df_block0_cleaned = df_block0.filter(
    col("`energy(kWh/hh)`").isNotNull() &
    (col("`energy(kWh/hh)`") != "Null") &
    (col("`energy(kWh/hh)`") != "null")
) \
.withColumn("energy_double", col("`energy(kWh/hh)`").cast("double")) \
.withColumn("timestamp", to_timestamp("tstp")) \
.withColumn("hour_time", date_trunc("hour", "timestamp"))

# Format timestamp data cuaca
df_weather_formatted = df_weather.withColumn("hour_time", to_timestamp("time"))

# Agregasi data per jam
df_block0_hourly = df_block0_cleaned.groupBy("hour_time").agg(
    _sum("energy_double").alias("sum_energy"),
    count("LCLid").alias("count_lcl")
)

# Penggabungan
df_merged = df_block0_hourly.join(df_weather_formatted, on="hour_time", how="inner")

# menampilkan 10 data pertama
df_merged.show(10)

+-------------------+------------------+---------+----------+-----------+-----------+-------------------+--------+--------+-------------------+---------+----------+-------------------+--------+--------------------+
|          hour_time|        sum_energy|count_lcl|visibility|windBearing|temperature|               time|dewPoint|pressure|apparentTemperature|windSpeed|precipType|               icon|humidity|             summary|
+-------------------+------------------+---------+----------+-----------+-----------+-------------------+--------+--------+-------------------+---------+----------+-------------------+--------+--------------------+
|2012-10-20 00:00:00|            21.535|       82|      4.63|        282|      11.91|2012-10-20 00:00:00|    11.0| 1010.75|              11.91|     2.65|      rain|partly-cloudy-night|    0.94|       Mostly Cloudy|
|2012-11-16 16:00:00| 55.79100000000001|       98|      1.08|        123|       8.59|2012-11-16 16:00:00|    7.78| 1012.54|               6.